# Homework: AI Tools, AI Agents, and Skills
In this homework, we will use AI to make a look-ahead schedule and answer questions about a construction project with tools, an AI agent, and a skill.

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line. Paste your Gemini API key from MP5 into *api_key* (or leave it empty to use the Colab secret GEMINI_API_KEY).
#@markdown Without a key, every step runs with its default settings (stored outputs); editing the agent's system prompt or a tool description needs the key.
api_key = "" #@param {type:"string"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp8_agents", "aec_agent"
FOLDERS = ["mp8_agents"]                      # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
try:
    import google.genai
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], capture_output=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_agent import lab
model = "gemini-3.5-flash-lite"
lab.setup(api_key=api_key, model=model)


## Part 1 · Tools: Code Execution and Tool Calling
In this part, we will explore two kinds of tools: code that HokieAI writes and runs by itself, and tools written for the Gemini agent, which it chooses from to answer the project questions Q4 to Q6.

![Code execution and tool calling](https://raw.githubusercontent.com/Haolan-Zhang/CEM4644/master/mp8_agents/figures/codeexec_hw.png)

- **search_spec**: searches the specification.
- **calculator**: exact arithmetic.
- **lookup_log**: reads the submittal log or the schedule.

In [ ]:
#@title ▶ Step 1a · Look-Ahead Schedule: A Gantt Chart { display-mode: "form" }
#@markdown Run the cell and follow the steps: HokieAI computes each activity's duration and start day, draws a Gantt chart, and marks in red the concrete placements whose mix design is not approved.
#@markdown Look-ahead schedule: the near-term part of the project schedule that the field team plans from.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Use your Python tool to make a look-ahead schedule from the two attached files, schedule.csv and submittal_log.csv.
1. For each activity, the duration in working days = quantity / daily output, rounded up to a whole day.
2. {sequence}
3. Draw a Gantt chart: one horizontal bar per activity, from its start day to its finish day, with working days on the x-axis. Color a bar red if the activity places concrete (its description starts with "Place") and the mix-design submittal for its element in submittal_log.csv is not Approved; color the other bars gray.
4. After the chart, write the schedule as plain text in your reply (not inside the chart image), one line per activity, in this format (the last column: YES if the bar is red, otherwise NO):
A-110 | <start day> | <finish day> | <YES or NO>"""
steps_text = """1. Download {files} with the buttons below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach both files, and paste the prompt. Look at the chart and open the code HokieAI wrote.
3. Copy HokieAI's output with the **copy button** below its reply, and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**Look-ahead schedule:** {right} of {n} activities with the correct start day, finish day, and flag."""
chart_title_text = """Ground truth: the look-ahead schedule (red = the mix design for the element is not approved)"""
axis_text = """working day"""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Activity | Ground truth (start – finish, red) | HokieAI's output | Correct"""
lab.lookahead(prompt_text=prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, chart_title_text=chart_title_text, axis_text=axis_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 1b · Look at the Project Documents { display-mode: "form" }
#@markdown Choose a document to see what the tools read.
document = "specification (Section 03 30 00)" #@param ["specification (Section 03 30 00)", "submittal log", "schedule"]
lab.show_document(document)


In [ ]:
#@title ▶ Step 1c · Run the Agent with the Tools You Choose { display-mode: "form" }
#@markdown Tick the tools and run the cell; each run is added to the table. Try no tools, each tool alone, and all three.
#@markdown UNKNOWN: the agent said it could not find the value. Wrong: it guessed.
search_spec = True #@param {type:"boolean"}
calculator = True #@param {type:"boolean"}
lookup_log = True #@param {type:"boolean"}
questions = ["Q4", "Q5", "Q6"]          # the project questions the agent answers (Q1-Q3 were in class)
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
system_text = """You are an AI agent that answers questions about the CEM4644 practice project, a fictional building project. Project information (specification requirements, submittals, schedule) is available only through your tools. Do not guess a project value: if you cannot find it, answer UNKNOWN.
End your reply with the final answer on its own line, in this format: FINAL: <answer>
If the question asks for several quantities, give one FINAL line per quantity, for example: FINAL: Q1 = 832 CMU"""
search_spec_text = """Searches the project specification (Section 03 30 00, cast-in-place concrete) and returns the lines that best match the search words, for example the requirements for one element."""
calculator_text = """Evaluates an arithmetic expression exactly and returns the result. Supports + - * / ** ( ), ceil, floor, round, sqrt, min and max, for example: ceil(212 / 48)."""
lookup_log_text = """Returns the rows of a project log that contain the search text: log = submittals (the submittal log: status and the mix of each submittal) or schedule (the schedule: quantity and daily output of each activity)."""
result_text = """**Run {run} · tools: {tools}**
**Correct:** {right} of {n}   **UNKNOWN:** {unknown}   **Wrong:** {wrong}"""
none_text = """no tools"""
table_text = """Ground truth vs. agent output (this run):"""
columns_text = """Question | Ground truth | Agent output | Result | Tools called"""
runs_text = """All runs so far:"""
runs_columns_text = """Run | Tools | Correct | UNKNOWN | Wrong"""
lab.agent_tools(search_spec, calculator, lookup_log, questions, system_text=system_text, search_spec_text=search_spec_text, calculator_text=calculator_text, lookup_log_text=lookup_log_text, result_text=result_text, none_text=none_text, table_text=table_text, columns_text=columns_text, runs_text=runs_text, runs_columns_text=runs_columns_text)


In [ ]:
#@title ▶ Step 1d · Inspect the Tool Calls { display-mode: "form" }
#@markdown Choose a question to see, for each run of Step 1c, the tools the agent called, with their inputs and outputs.
question = "Q6. Does the target slump of the mix in submittal 03 30 00-05 meet this project's maximum slump for its element? Answer YES or NO." #@param ["Q4. A continuous footing is 187 ft long, 2 ft 7 in wide, and 1 ft 2 in deep. How many cubic yards of concrete are needed, including 7 % waste?", "Q5. What is the review status of submittal 03 30 00-03?", "Q6. Does the target slump of the mix in submittal 03 30 00-05 meet this project's maximum slump for its element? Answer YES or NO."]
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
run_text = """**Run {run}**
tools: {tools}"""
question_label_text = """question"""
call_label_text = """tool call"""
output_label_text = """tool output"""
answer_label_text = """answer"""
empty_text = """No runs yet: run Step 1c first."""
lab.show_trace(question, run_text=run_text, question_label_text=question_label_text, call_label_text=call_label_text, output_label_text=output_label_text, answer_label_text=answer_label_text, empty_text=empty_text)


### 📝 Report question 1
> From Step 1a: Include HokieAI's Gantt chart and the score. Which activities did it mark red, and did any start or finish day differ from the ground truth? Open the code it wrote: which step went wrong, if any?

> Would you trust a look-ahead schedule that HokieAI computed with its own code, and what would you check before sending it to the field team?

### 📝 Report question 2
> From Step 1c: Include the table of runs. For each of Q4, Q5, and Q6, report which tools the agent needed to answer it correctly.

> From Step 1d: For question Q6, describe the agent's steps with all three tools. Without the tools it needed, did the agent answer UNKNOWN or guess?

> For which project tasks would you write a tool for an agent (as in Step 1c), and for which would you let it write its own code (as in Step 1a)?

## Part 2 · AI Agents: Your Own HokieAI Agent
A HokieAI agent is instructions plus knowledge sources (files it can read). In this part, we will build one with the project files and ask it all six project questions: Q1 to Q3 from the in-class activities and Q4 to Q6 from Part 1.

In [ ]:
#@title ▶ Step 2a · Create Your HokieAI Agent { display-mode: "form" }
#@markdown Run the cell and follow the steps; edit the instructions in the box before pasting them.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
instructions_text = """You answer questions about the CEM4644 practice project, a fictional building project, using only the attached files: the specification excerpt (spec_03_30_00.txt), the submittal log (submittal_log.csv), and the schedule (schedule.csv). Show each value you look up and each calculation. Do not guess a project value: if the files do not contain it, answer UNKNOWN."""
create_steps_text = """1. Download the three project files with the buttons below.
2. In HokieAI ({url}), open **Explore Agents** and click *Create* next to *My Agents*; enter a name, for example *CEM4644 project agent*.
3. Under **Agent Instructions → Specific**, paste the instructions below.
4. Under **Knowledge Sources → Add → Upload Files**, upload the three files.
5. Click *Create*."""
lab.create_agent(instructions_text=instructions_text, create_steps_text=create_steps_text)


In [ ]:
#@title ▶ Step 2b · Ask Your HokieAI Agent { display-mode: "form" }
#@markdown Run the cell after Step 2a and follow the steps. The table compares your agent with the Gemini agent's runs of Step 1c on Q4 to Q6.
questions = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"]          # the project questions your agent answers
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Answer the questions below. Reply with one line per question and nothing else, in this format:
Q1 | <answer>

Questions:
{questions}"""
steps_text = """1. Open your agent in HokieAI ({url}) and start a new chat with it.
2. Copy the prompt below, paste it into the chat, and send.
3. Copy the agent's output with the **copy button** below its reply, and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth and with the Gemini agent's runs in Step 1c."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**Your HokieAI agent:** {right} of {n} correct, {unknown} UNKNOWN, {wrong} wrong."""
table_text = """Ground truth vs. your HokieAI agent's output:"""
columns_text = """Question | Ground truth | HokieAI agent output | Result"""
compare_text = """Your HokieAI agent vs. the Gemini agent's runs in Step 1c (on the questions both answered):"""
hokieai_row_text = """HokieAI agent ({questions})"""
gemini_row_text = """Gemini agent, run {run} ({tools})"""
compare_columns_text = """Agent | Correct | UNKNOWN | Wrong"""
lab.hokieai_agent(questions, prompt_text=prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text, compare_text=compare_text, hokieai_row_text=hokieai_row_text, gemini_row_text=gemini_row_text, compare_columns_text=compare_columns_text)


### 📝 Report question 3
> From Step 2b: Report your agent's correct, UNKNOWN, and wrong answers on all six questions, include the tables, and compare it on Q4 to Q6 with the Gemini agent's run with all three tools.

> Which instructions did you change, and what changed in the output? For one wrong or UNKNOWN answer, explain from the output which value or calculation went wrong.

> What would you need to see before you let an agent answer project questions for your team?

## Part 3 · Skills: A Schedule Skill of Your Own
In this part, we will write a skill for a scheduling job of your choice that ends in a Gantt chart, then use it when the look-ahead schedule of Step 1a changes. You can ask HokieAI to help you develop the skill.

![Your skill, used on the schedule with a change you choose](https://raw.githubusercontent.com/Haolan-Zhang/CEM4644/master/mp8_agents/figures/schedule_skill.png)

In [ ]:
#@title ▶ Step 3a · Write Your Schedule Skill { display-mode: "form" }
#@markdown Run the cell and create your skill in HokieAI; there is no single correct skill. Choose the job it does for your team, for example:
#@markdown Subcontractor look-ahead: the next two weeks for one trade, with what it needs on site each day.
#@markdown Crew histogram: the crews needed each day, under the chart, with the days that need more crews than are available marked.
#@markdown Pour plan: the concrete trucks to order and the pump to book for each placement day.
#@markdown Meeting agenda: the weekly coordination meeting items read from the chart.
#@markdown Copy the required rule into the skill's body. You can ask HokieAI in a chat to draft or improve the skill, then edit it.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
steps_text = """1. Choose a job your skill does for your team; its output must include a Gantt chart.
2. In HokieAI ({url}), open **Account → My Skills → Create Skill**, and write the skill's name, its description (what HokieAI reads to decide when to use the skill), and its body (your rules, in Markdown).
3. Copy the required rule below into the body, so the notebook can score the dates.
4. Switch *Always On* off and click *Create Skill*.
You can ask HokieAI in a chat to help develop the skill: describe the job and your rules, ask it to draft the body, and edit the draft."""
required_text = """At the end of the reply, write the schedule as plain text (not inside the chart image), one line per activity:
<activity> | <start day> | <finish day>"""
required_label_text = """required rule"""
lab.schedule_skill(steps_text=steps_text, required_text=required_text, required_label_text=required_label_text)


In [ ]:
#@title ▶ Step 3b · Use Your Skill When the Schedule Changes { display-mode: "form" }
#@markdown Choose a change, run the cell, and follow the steps. The notebook recomputes the schedule for the change and scores the start and finish days.
#@markdown An activity starts late: *activity* cannot start before *day* (for example, its mix design is approved late).
#@markdown No work on some days: no work from *day* to *last_day* (for example, rain).
#@markdown One concrete placement at a time: concrete placements cannot overlap (for example, one pump on site).
#@markdown If the output does not follow your skill, revise the skill in **My Skills** and run the step again.
change = "an activity starts late" #@param ["an activity starts late", "no work on some days", "one concrete placement at a time", "no change"]
activity = "A-210" #@param ["A-110", "A-120", "A-210", "A-310", "A-410", "A-510"]
day = 10 #@param {type:"integer"}
last_day = 11 #@param {type:"integer"}
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Use my skill on the two attached files, schedule.csv and submittal_log.csv: make the look-ahead schedule with the change below.
The duration of each activity in working days = quantity / daily output, rounded up to a whole day.
{sequence}
{change}"""
late_start_text = """Change: {activity} cannot start before day {day}. Every activity after it moves as the sequence requires."""
no_work_text = """Change: no work is possible on days {day} to {last_day} (for example, rain). Keep numbering the days 1, 2, 3, ..., including these days. An activity cannot start on them; an activity in progress stops on them and continues after them, so its duration counts only the days with work."""
one_placement_text = """Change: only one concrete placement (an activity whose description starts with "Place") can be in progress on any day. Schedule the activities in the order they are listed in schedule.csv; each concrete placement starts on the first day after its predecessor finishes on which it can run its whole duration without overlapping a concrete placement listed before it."""
none_text = """Change: none."""
steps_text = """1. Download {files} with the buttons below and copy the request.
2. In HokieAI ({url}), start a new chat, turn on your skill from the chat menu, attach both files, and paste the request.
3. Copy HokieAI's output with the **copy button** below its reply, and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**Changed schedule:** {right} of {n} activities with the correct start day and finish day."""
chart_title_text = """Ground truth: the schedule before and after the change"""
before_text = """before the change"""
after_text = """after the change"""
off_text = """no work"""
axis_text = """day"""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Activity | Ground truth (start – finish) | HokieAI's output | Correct"""
lab.schedule_change(change, activity, day, last_day, prompt_text=prompt_text, late_start_text=late_start_text, no_work_text=no_work_text, one_placement_text=one_placement_text, none_text=none_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, chart_title_text=chart_title_text, before_text=before_text, after_text=after_text, off_text=off_text, axis_text=axis_text, table_text=table_text, columns_text=columns_text)


### 📝 Report question 4
> From Step 3a: Include your skill (name, description, and body) and say what job it does. If HokieAI helped you write it, what did you change in its draft?

> From Step 3b: Include HokieAI's output (the Gantt chart and what your skill adds), the change you chose, and the score. Did the output follow every rule of your skill? If you revised the skill, what did you change, and did it help?

> Would your field team use this output? What would they still check by hand before using it?

### Data and model sources
- Practice project (specification excerpt, submittal log, schedule): made for CEM4644 (fictional, not for construction); the specification excerpt follows the format of UFGS 03 30 00, U.S. Department of Defense, public domain, https://www.wbdg.org/dod/ufgs.
- AI model: Gemini (gemini-3.5-flash-lite), Google, via the Gemini API with your own key.
- Generative AI: GPT models via HokieAI (Virginia Tech).